In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/unified-dataset/final_dataset.json


In [2]:
!pip install torch
!pip install transformers
!pip install tree_sitter==0.21.3 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [3]:
!pip install tqdm

In [4]:
# DFG_java.py
from tree_sitter import Language, Parser

def DFG_java(root_node,index_to_code,states):
    assignment=['assignment_expression']
    def_statement=['variable_declarator']
    increment_statement=['update_expression']
    if_statement=['if_statement','else']
    for_statement=['for_statement']
    enhanced_for_statement=['enhanced_for_statement']
    while_statement=['while_statement']
    do_first_statement=[]    
    states=states.copy()
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        idx,code=index_to_code[(root_node.start_point,root_node.end_point)]
        if root_node.type==code:
            return [],states
        elif code in states:
            return [(code,idx,'comesFrom',[code],states[code].copy())],states
        else:
            if root_node.type=='identifier':
                states[code]=[idx]
            return [(code,idx,'comesFrom',[],[])],states
    elif root_node.type in def_statement:
        name=root_node.child_by_field_name('name')
        value=root_node.child_by_field_name('value')
        DFG=[]
        if value is None:
            indexs=tree_to_variable_index(name,index_to_code)
            for index in indexs:
                idx,code=index_to_code[index]
                DFG.append((code,idx,'comesFrom',[],[]))
                states[code]=[idx]
            return sorted(DFG,key=lambda x:x[1]),states
        else:
            name_indexs=tree_to_variable_index(name,index_to_code)
            value_indexs=tree_to_variable_index(value,index_to_code)
            temp,states=DFG_java(value,index_to_code,states)
            DFG+=temp            
            for index1 in name_indexs:
                idx1,code1=index_to_code[index1]
                for index2 in value_indexs:
                    idx2,code2=index_to_code[index2]
                    DFG.append((code1,idx1,'comesFrom',[code2],[idx2]))
                states[code1]=[idx1]   
            return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in assignment:
        left_nodes=root_node.child_by_field_name('left')
        right_nodes=root_node.child_by_field_name('right')
        DFG=[]
        temp,states=DFG_java(right_nodes,index_to_code,states)
        DFG+=temp            
        name_indexs=tree_to_variable_index(left_nodes,index_to_code)
        value_indexs=tree_to_variable_index(right_nodes,index_to_code)        
        for index1 in name_indexs:
            idx1,code1=index_to_code[index1]
            for index2 in value_indexs:
                idx2,code2=index_to_code[index2]
                DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
            states[code1]=[idx1]   
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in increment_statement:
        DFG=[]
        indexs=tree_to_variable_index(root_node,index_to_code)
        for index1 in indexs:
            idx1,code1=index_to_code[index1]
            for index2 in indexs:
                idx2,code2=index_to_code[index2]
                DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
            states[code1]=[idx1]
        return sorted(DFG,key=lambda x:x[1]),states   
    elif root_node.type in if_statement:
        DFG=[]
        current_states=states.copy()
        others_states=[]
        flag=False
        tag=False
        if 'else' in root_node.type:
            tag=True
        for child in root_node.children:
            if 'else' in child.type:
                tag=True
            if child.type not in if_statement and flag is False:
                temp,current_states=DFG_java(child,index_to_code,current_states)
                DFG+=temp
            else:
                flag=True
                temp,new_states=DFG_java(child,index_to_code,states)
                DFG+=temp
                others_states.append(new_states)
        others_states.append(current_states)
        if tag is False:
            others_states.append(states)
        new_states={}
        for dic in others_states:
            for key in dic:
                if key not in new_states:
                    new_states[key]=dic[key].copy()
                else:
                    new_states[key]+=dic[key]
        for key in new_states:
            new_states[key]=sorted(list(set(new_states[key])))
        return sorted(DFG,key=lambda x:x[1]),new_states
    elif root_node.type in for_statement:
        DFG=[]
        for child in root_node.children:
            temp,states=DFG_java(child,index_to_code,states)
            DFG+=temp
        flag=False
        for child in root_node.children:
            if flag:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp                
            elif child.type=="local_variable_declaration":
                flag=True
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in enhanced_for_statement:
        name=root_node.child_by_field_name('name')
        value=root_node.child_by_field_name('value')
        body=root_node.child_by_field_name('body')
        DFG=[]
        for i in range(2):
            temp,states=DFG_java(value,index_to_code,states)
            DFG+=temp       
            name_indexs=tree_to_variable_index(name,index_to_code)
            value_indexs=tree_to_variable_index(value,index_to_code)        
            for index1 in name_indexs:
                idx1,code1=index_to_code[index1]
                for index2 in value_indexs:
                    idx2,code2=index_to_code[index2]
                    DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
                states[code1]=[idx1]   
            temp,states=DFG_java(body,index_to_code,states)
            DFG+=temp                       
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in while_statement:  
        DFG=[]
        for i in range(2):
            for child in root_node.children:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp    
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states        
    else:
        DFG=[]
        for child in root_node.children:
            if child.type in do_first_statement:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp
        for child in root_node.children:
            if child.type not in do_first_statement:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp
        
        return sorted(DFG,key=lambda x:x[1]),states

In [5]:
def DFG_c(root_node, index_to_code, states):
    assignment = ['assignment_expression']
    # In C, declarations are 'declaration', and the name/value is in 'init_declarator'
    def_statement = ['declaration'] 
    increment_statement = ['update_expression']
    if_statement = ['if_statement', 'else']
    for_statement = ['for_statement']
    while_statement = ['while_statement']
    do_first_statement = []
    
    states = states.copy()
    
    # Base case: Token
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        idx, code = index_to_code[(root_node.start_point, root_node.end_point)]
        if root_node.type == code:
            return [], states
        elif code in states:
            return [(code, idx, 'comesFrom', [code], states[code].copy())], states
        else:
            if root_node.type == 'identifier':
                states[code] = [idx]
            return [(code, idx, 'comesFrom', [], [])], states
            
    # Variable Declaration (Adapted for C structure: declaration -> init_declarator)
    elif root_node.type in def_statement:
        DFG = []
        for child in root_node.children:
            if child.type == 'init_declarator':
                name = child.child_by_field_name('declarator')
                value = child.child_by_field_name('value')
                
                if value is None:
                    indexs = tree_to_variable_index(name, index_to_code)
                    for index in indexs:
                        idx, code = index_to_code[index]
                        DFG.append((code, idx, 'comesFrom', [], []))
                        states[code] = [idx]
                else:
                    name_indexs = tree_to_variable_index(name, index_to_code)
                    value_indexs = tree_to_variable_index(value, index_to_code)
                    temp, states = DFG_c(value, index_to_code, states)
                    DFG += temp
                    for index1 in name_indexs:
                        idx1, code1 = index_to_code[index1]
                        for index2 in value_indexs:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'comesFrom', [code2], [idx2]))
                        states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states

    # Assignment
    elif root_node.type in assignment:
        left_nodes = root_node.child_by_field_name('left')
        right_nodes = root_node.child_by_field_name('right')
        DFG = []
        temp, states = DFG_c(right_nodes, index_to_code, states)
        DFG += temp
        name_indexs = tree_to_variable_index(left_nodes, index_to_code)
        value_indexs = tree_to_variable_index(right_nodes, index_to_code)
        for index1 in name_indexs:
            idx1, code1 = index_to_code[index1]
            for index2 in value_indexs:
                idx2, code2 = index_to_code[index2]
                DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
            states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    # Increment
    elif root_node.type in increment_statement:
        DFG = []
        indexs = tree_to_variable_index(root_node, index_to_code)
        for index1 in indexs:
            idx1, code1 = index_to_code[index1]
            for index2 in indexs:
                idx2, code2 = index_to_code[index2]
                DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
            states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    # Standard Control Flow (Recursion)
    else:
        DFG = []
        for child in root_node.children:
            if child.type in do_first_statement:
                temp, states = DFG_c(child, index_to_code, states)
                DFG += temp
        for child in root_node.children:
            if child.type not in do_first_statement:
                temp, states = DFG_c(child, index_to_code, states)
                DFG += temp
        return sorted(DFG, key=lambda x: x[1]), states

In [6]:
def clean_and_wrap_code(code_snippet, lang='java', mode='auto'):
    """
    Wraps code based on the requested mode.
    modes: 'auto', 'method', 'class', 'raw'
    """
    code_snippet = code_snippet.strip()
    
    if lang == 'java':
        if mode == 'raw':
            return code_snippet
        elif mode == 'class':
            return f"public class DummyClass {{ {code_snippet} }}"
        elif mode == 'method':
            return f"public class DummyClass {{ public void dummyMethod() {{ {code_snippet} }} }}"
        else: # auto
            if "class " in code_snippet and "{" in code_snippet:
                return code_snippet
            if any(x in code_snippet for x in ["public ", "private ", "protected "]):
                return f"public class DummyClass {{ {code_snippet} }}"
            return f"public class DummyClass {{ public void dummyMethod() {{ {code_snippet} }} }}"

    elif lang == 'c':
        if mode == 'raw' or code_snippet.startswith("#"):
            return code_snippet
        if mode == 'method':
            return f"void dummy_function() {{ {code_snippet} }}"
        # Default C wrapper
        if re.search(r'^\s*(?:[\w\s\*]+)\s+[\w\*]+\s*\([^\)]*\)\s*\{', code_snippet, re.DOTALL):
            return code_snippet
        return f"void dummy_function() {{ {code_snippet} }}"
        
    return code_snippet

In [7]:
import os
import shutil
from tree_sitter import Language

# --- 1. CLEANUP ---
print("Cleaning up old files...")
folders_to_remove = ['build', 'tree-sitter-java', 'tree-sitter-c']
for folder in folders_to_remove:
    if os.path.exists(folder):
        print(f"  Deleting {folder}...")
        shutil.rmtree(folder)

# --- 2. PREPARE DIRECTORY ---
# This was the missing step causing your error!
print("Creating 'build' directory...")
os.makedirs('build', exist_ok=True) 

# --- 3. CLONE REPOS (Hybrid Setup) ---
JAVA_REPO = 'tree-sitter-java'
C_REPO = 'tree-sitter-c'
BUILD_PATH = 'build/my-languages.so'

# A. Java: Latest (Master) - Fixes DFG empty issues
print(f"Cloning {JAVA_REPO} (Latest)...")
os.system("git clone https://github.com/tree-sitter/tree-sitter-java")

# B. C: Pinned (v0.20.6) - Fixes ABI Version 15 crash
print(f"Cloning {C_REPO} (Pinned to v0.20.6)...")
os.system("git clone --branch v0.20.6 --depth 1 https://github.com/tree-sitter/tree-sitter-c")

# --- 4. BUILD ---
print(f"Compiling languages to {BUILD_PATH}...")
try:
    Language.build_library(
        BUILD_PATH,
        [JAVA_REPO, C_REPO]
    )
    print("\nSUCCESS! Parsers built successfully.")
    print("You can now run your main pipeline script.")
except Exception as e:
    print(f"\nFAIL: {e}")

Cleaning up old files...
Creating 'build' directory...
Cloning tree-sitter-java (Latest)...


Cloning into 'tree-sitter-java'...


Cloning tree-sitter-c (Pinned to v0.20.6)...


Cloning into 'tree-sitter-c'...
Note: switching to 'a2b7bac3b313efbaa683d9a276ff63cdc544d960'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

/usr/local/lib/python3.11/dist-packages/tree_sitter/__init__.py:36: FutureWarning: Language.build_library is deprecated. Use the new bindings instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


Compiling languages to build/my-languages.so...

SUCCESS! Parsers built successfully.
You can now run your main pipeline script.


In [8]:
#utils.py
def tree_to_token_index(root_node):
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        return [(root_node.start_point,root_node.end_point)]
    else:
        code_tokens=[]
        for child in root_node.children:
            code_tokens+=tree_to_token_index(child)
        return code_tokens
    
def tree_to_variable_index(root_node,index_to_code):
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        index=(root_node.start_point,root_node.end_point)
        _,code=index_to_code[index]
        if root_node.type!=code:
            return [(root_node.start_point,root_node.end_point)]
        else:
            return []
    else:
        code_tokens=[]
        for child in root_node.children:
            code_tokens+=tree_to_variable_index(child,index_to_code)
        return code_tokens    

def index_to_code_token(index,code):
    start_point=index[0]
    end_point=index[1]
    if start_point[0]==end_point[0]:
        s=code[start_point[0]][start_point[1]:end_point[1]]
    else:
        s=""
        s+=code[start_point[0]][start_point[1]:]
        for i in range(start_point[0]+1,end_point[0]):
            s+=code[i]
        s+=code[end_point[0]][:end_point[1]]   
    return s

In [9]:
def process_single_item(code_snippet, lang, parser):
    # We will try these modes in order. 
    # 'method' is usually best for statements (recovers variables).
    # 'class' is best for declarations.
    if lang == 'java':
        modes_to_try = ['method', 'class', 'auto'] 
    else:
        modes_to_try = ['method', 'auto']

    best_result = None
    max_nodes = -1

    for mode in modes_to_try:
        try:
            # 1. Wrap
            wrapped_code = clean_and_wrap_code(code_snippet, lang, mode)
            
            # 2. Parse
            tree = parser.parse(bytes(wrapped_code, 'utf8'))
            root_node = tree.root_node
            
            # Quick check: If the tree is mostly ERRORs, this wrapper is bad.
            # (Optional optimization: skip DFG if root.has_error is True)

            # 3. Indexing
            tokens_index = tree_to_token_index(root_node)
            code_lines = wrapped_code.split('\n')
            index_to_code = {}
            
            for idx, (start, end) in enumerate(tokens_index):
                string_val = index_to_code_token((start, end), code_lines)
                index_to_code[(start, end)] = ((start, end), string_val)

            # 4. Extract DFG
            dfg_function = DFG_java if lang == 'java' else DFG_c
            DFG, _ = dfg_function(root_node, index_to_code, {})
            
            # 5. Score this attempt
            # We prefer the result with the MOST nodes found.
            if len(DFG) > max_nodes:
                max_nodes = len(DFG)
                best_result = {
                    "code": wrapped_code,
                    "dfg": DFG
                }
                
            # If we got a good result, stop trying other modes? 
            # No, keep trying to ensure we find the absolute best.
            
        except Exception:
            continue

    # Return the best result found (or an empty one if all failed)
    if best_result is None:
        return {"code": code_snippet, "dfg": []}
        
    return best_result

In [10]:
dataset = '/kaggle/input/unified-dataset/final_dataset.json'

In [11]:
import json
import traceback
from tqdm import tqdm

def main():
    # ---------------------------------------------------------
    # 1. Configuration
    # ---------------------------------------------------------
    INPUT_FILE = '/kaggle/input/unified-dataset/final_dataset.json'
    OUTPUT_FILE = 'dataset_graphcodebert.jsonl' # Saves to /kaggle/working/
    
    print(f"--- Starting Pipeline ---")
    
    # ---------------------------------------------------------
    # 2. Setup & Loading
    # ---------------------------------------------------------
    print("1. Setting up Tree-sitter parsers...")
    try:
        LIB_PATH = 'build/my-languages.so' 
        # Load the languages directly
        JAVA_LANG = Language(LIB_PATH, 'java')
        C_LANG = Language(LIB_PATH, 'c')
        parser = Parser()
    except Exception as e:
        print(f"CRITICAL ERROR during parser setup: {e}")
        return

    print(f"2. Loading dataset from {INPUT_FILE}...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f_in:
            dataset = json.load(f_in)
    except FileNotFoundError:
        print(f"Error: File not found at {INPUT_FILE}")
        return
    except json.JSONDecodeError:
        print("Error: Failed to parse JSON. File might be corrupt or huge.")
        return
        
    print(f"   Successfully loaded {len(dataset)} items.")

    # ---------------------------------------------------------
    # 3. Diagnostic Phase (Test first 5 items)
    # ---------------------------------------------------------
    print("\n3. Running Diagnostic Check on first 5 items...")
    test_batch = dataset[:5]
    failures = 0
    
    for i, entry in enumerate(test_batch):
        try:
            # Check for dictionary type
            if not isinstance(entry, dict):
                print(f"   [Item {i}] FAILED: Entry is type {type(entry)}, expected dict.")
                failures += 1
                continue
                
            code_snippet = entry.get('code', '')
            if not code_snippet:
                print(f"   [Item {i}] WARNING: Code field is empty.")
            
            # Mock language detection for test
            filename = entry.get('filename', '')
            if "LVDAndro" in filename or "public " in code_snippet or ".java" in filename:
                lang = 'java'
                parser.set_language(JAVA_LANG)
            else:
                lang = 'c'
                parser.set_language(C_LANG)
            
            # Run pipeline
            result = process_single_item(code_snippet, lang, parser)
            
            if result and len(result['dfg']) > 0:
                print(f"   [Item {i}] SUCCESS ({lang}): Generated DFG with {len(result['dfg'])} nodes.")
            else:
                print(f"   [Item {i}] WARNING: Parsed successfully but DFG is empty.")
                
        except Exception as e:
            print(f"   [Item {i}] CRITICAL FAIL:")
            traceback.print_exc() # Prints full error to console
            failures += 1

    if failures > 0:
        print(f"\n(!) Diagnostic found {failures} errors in the first 5 items.")
        print("    Proceeding with caution... (check logs above if many fail)\n")
    else:
        print("\n(+) Diagnostic passed. Starting full processing.\n")

    # ---------------------------------------------------------
    # 4. Full Processing Loop
    # ---------------------------------------------------------
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        success_count = 0
        
        # Use tqdm for progress bar
        for entry in tqdm(dataset, desc="Extracting DFGs"):
            if not isinstance(entry, dict): continue
            
            code_snippet = entry.get('code', '')
            if not code_snippet: continue
            
            # Language Switch
            filename = entry.get('filename', '')
            if "LVDAndro" in filename or "public " in code_snippet or ".java" in filename:
                lang = 'java'
                parser.set_language(JAVA_LANG)
            else:
                lang = 'c'
                parser.set_language(C_LANG)
                
            # Process
            result = process_single_item(code_snippet, lang, parser)
            
            if result:
                # Add metadata back
                result['label'] = entry.get('label')
                result['filename'] = entry.get('filename')
                
                # Write line
                json_str = json.dumps(result, default=lambda x: list(x) if isinstance(x, tuple) else str(x))
                f_out.write(json_str + '\n')
                success_count += 1

    print(f"Processing complete.")
    print(f"Successfully processed {success_count}/{len(dataset)} items.")
    print(f"Saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

--- Starting Pipeline ---
1. Setting up Tree-sitter parsers...
2. Loading dataset from /kaggle/input/unified-dataset/final_dataset.json...


/usr/local/lib/python3.11/dist-packages/tree_sitter/__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


   Successfully loaded 199960 items.

3. Running Diagnostic Check on first 5 items...
   [Item 0] SUCCESS (java): Generated DFG with 64 nodes.
   [Item 1] SUCCESS (java): Generated DFG with 52 nodes.
   [Item 2] SUCCESS (c): Generated DFG with 181 nodes.
   [Item 3] SUCCESS (c): Generated DFG with 117 nodes.
   [Item 4] SUCCESS (c): Generated DFG with 58 nodes.

(+) Diagnostic passed. Starting full processing.



Extracting DFGs: 100%|██████████| 199960/199960 [13:46<00:00, 241.82it/s]


Processing complete.
Successfully processed 199960/199960 items.
Saved to: dataset_graphcodebert.jsonl


In [12]:
import json
from tqdm import tqdm

INPUT_FILE = '/kaggle/working/dataset_graphcodebert.jsonl'

print(f"Validating {INPUT_FILE}...")

invalid_lines = []
total_lines = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in tqdm(enumerate(f)):
        total_lines += 1
        line = line.strip()
        if not line: continue # Skip empty newlines
        
        try:
            # Try to parse the line as a JSON object
            data = json.loads(line)
            
            # Optional: Check if keys exist
            if "code" not in data or "dfg" not in data:
                print(f"Line {i}: Missing 'code' or 'dfg' keys.")
                invalid_lines.append(i)
                
        except json.JSONDecodeError as e:
            # This will trigger if a string is not closed properly
            print(f"Line {i} is BROKEN: {e}")
            invalid_lines.append(i)

if len(invalid_lines) == 0:
    print(f"\nSUCCESS! All {total_lines} lines are valid JSON strings.")
    print("The display issue in VS Code is just a rendering limit.")
else:
    print(f"\nFAIL: Found {len(invalid_lines)} broken lines.")

Validating /kaggle/working/dataset_graphcodebert.jsonl...


199960it [01:07, 2973.94it/s]


SUCCESS! All 199960 lines are valid JSON strings.
The display issue in VS Code is just a rendering limit.
